# NB2 — Analysis, Statistics & Results

**Stage 2 of 3.** Consumes **`outputs/_metadata_merged.csv`** (produced by NB1) plus the raw-derived intermediates `rich_features.csv` and `detector_robustness.csv` (also from NB1), and runs the full analysis: grouping (unsupervised, supervised, PCA), the dose-response tests, effect sizes, the Bayesian hierarchical model, consensus phenotyping, and the advanced robustness suite. Produces every results table and thesis figure.

**Prerequisite:** run NB1 first. Each section below is self-contained (re-reads its inputs), so sections can be re-run independently.

In [ ]:
# ==========================================================
# Inline helper: save each figure panel as its own single graph.
# Registered as an importable module so the analysis cells below can use
# `import figutil` with NO separate file -- this notebook is self-contained.
# ==========================================================
import sys, types, os
import matplotlib.pyplot as plt

def _explode(fig, prefix, outdir="../outputs/figures", short=None, dpi=150):
    """Save each content panel of `fig` as its own PNG (short title, tables skipped)."""
    if getattr(fig, "_suptitle", None) is not None:
        fig._suptitle.set_text("")                       # drop the big figure title
    axes = [a for a in fig.axes
            if (a.get_position().width * a.get_position().height) > 0.02]
    content = [a for a in axes if a.lines or a.patches or a.collections or a.images]  # skip text-only tables
    for i, a in enumerate(content):
        t = short[i] if (short and i < len(short)) else a.get_title().split("\n")[0]
        a.set_title(t or ("panel %d" % (i + 1)), fontsize=11, fontweight="bold")
    os.makedirs(outdir, exist_ok=True)
    fig.canvas.draw(); r = fig.canvas.get_renderer()
    saved = []
    for i, a in enumerate(content):
        ext = a.get_tightbbox(r).transformed(fig.dpi_scale_trans.inverted())
        name = "%s_%s.png" % (prefix, chr(97 + i))
        fig.savefig(os.path.join(outdir, name), bbox_inches=ext.expanded(1.02, 1.05), dpi=dpi)
        saved.append(name)
    plt.close(fig)
    print("  saved %d single-graph panels: %s" % (len(saved), ", ".join(saved)))
    return saved

figutil = types.ModuleType("figutil"); figutil.explode = _explode
sys.modules["figutil"] = figutil          # later `import figutil` cells resolve to this


## Descriptive statistics & Control-vs-Dox

*Per-parameter group means, the failed positive control (Control ≈ Dox), and omnibus tests.*  
<sub>source: `group_analysis.py`</sub>

In [ ]:
"""Treatment-group analysis of the ECG features.

Produces, per study (acute / chronic):
  1. a group summary table (mean +/- SD per parameter per group)  -- Roisin's Table 1 format
  2. one-way ANOVA per parameter (group), with Tukey HSD vs Control
  3. a non-parametric Kruskal-Wallis alongside, since cells are small
  4. dose-response across ethanolamine (Spearman, 0/1.6/16/160)
For the chronic study only (cells large enough):
  5. two-way ANOVA group x sex

Assumption checks (Shapiro normality, Levene equal-variance) are reported so the
reader can see when the parametric ANOVA is trustworthy and when the
Kruskal-Wallis result should be preferred.

Uses CLEAN recordings only (status OK, rr_cv <= 0.15). Parameters known to be
unreliable are flagged, not dropped.

Run:  python group_analysis.py
Out:  ../outputs/anova/*.csv
"""
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore")
OUT = "../outputs/anova"
os.makedirs(OUT, exist_ok=True)

GROUPS = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]
ETN_DOSE = {"Control": 0, "Dox": 0, "Dox+Etn1.6": 1.6, "Dox+Etn16": 16, "Dox+Etn160": 160}

# (column, label, unit, reliability note)
PARAMS = [
    ("heart_rate_bpm", "Heart rate", "bpm", ""),
    ("rr_mean_ms", "RR interval", "ms", ""),
    ("qt_ms", "QT interval", "ms", ""),
    ("qtc_ms", "QTc", "ms", ""),
    ("qrs_duration_ms", "QRS (R-FWHM)", "ms", "FLAG: 1 ms quantised; R-width not clinical QRS"),
    ("r_amplitude_mv", "R amplitude", "mV", ""),
    ("p_wave_amplitude_mv", "P amplitude", "mV", ""),
    ("q_wave_amplitude_mv", "Q amplitude", "mV", "FLAG: near noise floor (mice often lack Q)"),
    ("s_wave_amplitude_mv", "S amplitude", "mV", "FLAG: window under review"),
    ("t_wave_amplitude_mv", "T amplitude", "mV", "FLAG: window under review"),
    ("j_wave_amplitude_mv", "J amplitude", "mV", "FLAG: window under review"),
]


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def load():
    m = pd.read_csv("../outputs/_metadata_merged.csv")
    m["group"] = pd.Categorical(m["group"], categories=GROUPS, ordered=True)
    m = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()
    m["etn_dose"] = m["group"].map(ETN_DOSE).astype(float)
    return m


def summary_table(df, study):
    rows = []
    for col, label, unit, flag in PARAMS:
        row = {"parameter": label, "unit": unit}
        for gname in GROUPS:
            x = pd.to_numeric(df[df.group == gname][col], errors="coerce").dropna()
            row[gname] = f"{x.mean():.3f} +/- {x.std():.3f}" if len(x) else "-"
            row[f"{gname}_n"] = len(x)
        row["flag"] = flag
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/summary_{study}.csv", index=False)
    return out


def oneway(df, study):
    rows = []
    for col, label, unit, flag in PARAMS:
        groups = [pd.to_numeric(df[df.group == g][col], errors="coerce").dropna().values
                  for g in GROUPS]
        groups_nz = [g for g in groups if len(g) >= 2]
        if len(groups_nz) < 2:
            continue
        # parametric
        F, p_anova = stats.f_oneway(*groups_nz)
        # non-parametric
        H, p_kw = stats.kruskal(*groups_nz)
        # assumption checks
        allx = np.concatenate(groups_nz)
        _, p_norm = stats.shapiro(allx) if len(allx) >= 3 else (np.nan, np.nan)
        try:
            _, p_lev = stats.levene(*groups_nz)
        except Exception:
            p_lev = np.nan
        # Tukey vs Control
        sub = df[[col, "group"]].copy()
        sub[col] = pd.to_numeric(sub[col], errors="coerce")
        sub = sub.dropna()
        vs_ctrl = {}
        if sub.group.nunique() >= 2 and len(sub) > sub.group.nunique():
            try:
                tuk = pairwise_tukeyhsd(sub[col], sub["group"])
                td = pd.DataFrame(tuk.summary().data[1:], columns=tuk.summary().data[0])
                for _, r in td.iterrows():
                    if "Control" in (r["group1"], r["group2"]):
                        other = r["group2"] if r["group1"] == "Control" else r["group1"]
                        vs_ctrl[other] = float(r["p-adj"])
            except Exception:
                pass
        row = dict(parameter=label, unit=unit, F=round(F, 2), p_anova=round(p_anova, 4),
                   sig_anova=stars(p_anova), p_kruskal=round(p_kw, 4),
                   sig_kruskal=stars(p_kw),
                   normal=("yes" if p_norm > 0.05 else "NO") if p_norm == p_norm else "-",
                   equal_var=("yes" if p_lev > 0.05 else "NO") if p_lev == p_lev else "-",
                   flag=flag)
        for g in GROUPS[1:]:
            row[f"vsCtrl_{g}"] = (f"{stars(vs_ctrl[g])}(p={vs_ctrl[g]:.3f})"
                                  if g in vs_ctrl else "-")
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/oneway_{study}.csv", index=False)
    return out


def dose_response(df, study):
    # ethanolamine trend within the dox-treated arms (Dox=0, Etn1.6/16/160)
    dd = df[df.group != "Control"].copy()
    rows = []
    for col, label, unit, flag in PARAMS:
        x = pd.to_numeric(dd[col], errors="coerce")
        ok = x.notna()
        if ok.sum() < 5:
            continue
        rho, p = stats.spearmanr(dd["etn_dose"][ok], x[ok])
        rows.append(dict(parameter=label, unit=unit, spearman_rho=round(rho, 3),
                         p=round(p, 4), sig=stars(p), n=int(ok.sum()), flag=flag))
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/doseresponse_{study}.csv", index=False)
    return out


def twoway(df, study):
    rows = []
    for col, label, unit, flag in PARAMS:
        sub = df[[col, "group", "sex"]].copy()
        sub[col] = pd.to_numeric(sub[col], errors="coerce")
        sub = sub.dropna()
        sub["group"] = sub["group"].astype(str)
        if sub.group.nunique() < 2 or sub.sex.nunique() < 2 or len(sub) < 12:
            continue
        try:
            model = ols(f'Q("{col}") ~ C(group) * C(sex)', data=sub).fit()
            aov = sm.stats.anova_lm(model, typ=2)
            rows.append(dict(
                parameter=label, unit=unit,
                p_group=round(aov.loc["C(group)", "PR(>F)"], 4),
                p_sex=round(aov.loc["C(sex)", "PR(>F)"], 4),
                p_interaction=round(aov.loc["C(group):C(sex)", "PR(>F)"], 4),
                sig_group=stars(aov.loc["C(group)", "PR(>F)"]),
                sig_interaction=stars(aov.loc["C(group):C(sex)", "PR(>F)"]),
                flag=flag))
        except Exception as ex:
            rows.append(dict(parameter=label, unit=unit, p_group="err:%s" % type(ex).__name__))
    out = pd.DataFrame(rows)
    out.to_csv(f"{OUT}/twoway_{study}.csv", index=False)
    return out


def main():
    m = load()
    print(f"CLEAN recordings with metadata: {len(m)}")
    print(f"{'='*70}")
    for study in ["acute", "chronic"]:
        d = m[m.study == study]
        n_by_g = d.group.value_counts().reindex(GROUPS)
        print(f"\n########## STUDY: {study.upper()}  (n={len(d)}) ##########")
        print("group sizes:", {g: int(n_by_g[g]) for g in GROUPS})

        st = summary_table(d, study)
        print(f"\n--- Summary (mean +/- SD) ---")
        print(st[["parameter"] + GROUPS].to_string(index=False))

        ow = oneway(d, study)
        print(f"\n--- One-way ANOVA (group) + Kruskal-Wallis ---")
        cols = ["parameter", "p_anova", "sig_anova", "p_kruskal", "sig_kruskal", "normal", "equal_var"]
        print(ow[cols].to_string(index=False))

        dr = dose_response(d, study)
        print(f"\n--- Dose-response (Spearman, Etn 0->160 within dox arms) ---")
        print(dr[["parameter", "spearman_rho", "p", "sig", "n"]].to_string(index=False))

        if study == "chronic":
            tw = twoway(d, study)
            print(f"\n--- Two-way ANOVA (group x sex) ---")
            print(tw.to_string(index=False))
        else:
            print(f"\n--- Two-way ANOVA: SKIPPED for acute (cells 1-4, insufficient) ---")

    print(f"\n{'='*70}\nAll tables written to {OUT}/")


if __name__ == "__main__":
    main()

## All-parameter inferential tests

*Assumption-checked test path for every ECG parameter → all_tests_results.csv.*  
<sub>source: `all_tests_table.py`</sub>

In [ ]:
"""Master statistical table: every ECG parameter x every test, on the raw data.
Grouped by ethanolamine dose (No-Etn / 1.6 / 16 / 160), per study.
Tests: one-way ANOVA, Kruskal-Wallis, Jonckheere-Terpstra trend, Spearman trend,
       two-way ANOVA (dose x sex: dose / sex / interaction p).
Writes ../outputs/all_tests_results.csv and prints markdown tables.
"""
import numpy as np, pandas as pd
from itertools import combinations
from scipy import stats

m = pd.read_csv("../outputs/_metadata_merged.csv")
m = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()


def dose_of(g):
    if g in ("Control", "Dox"):
        return 0.0
    return {"Dox+Etn1.6": 1.6, "Dox+Etn16": 16.0, "Dox+Etn160": 160.0}[g]


m["dose"] = m["group"].map(dose_of)
m["sex"] = m["Sex"].str.lower()
ORDER = [0.0, 1.6, 16.0, 160.0]
DLAB = ["No-Etn", "1.6", "16", "160"]

PARAMS = [
    ("heart_rate_bpm", "Heart rate (bpm)"),
    ("rr_mean_ms", "RR interval (ms)"),
    ("qt_ms", "QT (ms)"),
    ("qtc_ms", "QTc (ms)"),
    ("qrs_duration_ms", "QRS (ms)"),
    ("r_amplitude_mv", "R-amp (mV)"),
    ("p_wave_amplitude_mv", "P-amp (mV)"),
    ("t_wave_amplitude_mv", "T-amp (mV)"),
    ("j_wave_amplitude_mv", "J-amp (mV)"),
    ("s_wave_amplitude_mv", "S-amp (mV)"),
]


def jonckheere(groups):
    k = len(groups); U = 0
    for i, j in combinations(range(k), 2):
        for a in groups[i]:
            for b in groups[j]:
                U += (b > a) + 0.5 * (b == a)
    ns = np.array([len(g) for g in groups]); N = ns.sum()
    if N < 3 or (ns == 0).any():
        return np.nan, np.nan
    mean = (N**2 - (ns**2).sum()) / 4.0
    var = (N**2 * (2*N + 3) - (ns**2 * (2*ns + 3)).sum()) / 72.0
    if var <= 0:
        return np.nan, np.nan
    z = (U - mean) / np.sqrt(var)
    return z, 2 * stats.norm.sf(abs(z))


def two_way(d, col):
    try:
        import statsmodels.formula.api as smf
        from statsmodels.stats.anova import anova_lm
        dd = d.dropna(subset=[col, "sex"]).copy()
        dd["dosef"] = dd["dose"].astype("category")
        aov = anova_lm(smf.ols(f"{col} ~ C(dosef) * C(sex)", data=dd).fit(), typ=2)
        return (aov.loc["C(dosef)", "PR(>F)"], aov.loc["C(sex)", "PR(>F)"],
                aov.loc["C(dosef):C(sex)", "PR(>F)"])
    except Exception:
        return (np.nan, np.nan, np.nan)


rows = []
for study in ["chronic", "acute"]:
    s = m[m.study == study]
    for col, name in PARAMS:
        if col not in s.columns:
            continue
        d = s.copy(); d[col] = pd.to_numeric(d[col], errors="coerce"); d = d.dropna(subset=[col])
        groups = [d[d.dose == x][col].values for x in ORDER]
        if any(len(g) < 2 for g in groups):
            continue
        F, p_anova = stats.f_oneway(*groups)
        H, p_kw = stats.kruskal(*groups)
        zJT, p_jt = jonckheere(groups)
        rho, p_sp = stats.spearmanr(d["dose"], d[col])
        p_d, p_s, p_i = two_way(d, col)
        stats_row = {
            "study": study, "parameter": name,
            "No-Etn (mean±SD)": f"{groups[0].mean():.3f}±{groups[0].std():.3f}",
            "1.6": f"{groups[1].mean():.3f}±{groups[1].std():.3f}",
            "16": f"{groups[2].mean():.3f}±{groups[2].std():.3f}",
            "160": f"{groups[3].mean():.3f}±{groups[3].std():.3f}",
            "ANOVA p": round(p_anova, 3),
            "Kruskal p": round(p_kw, 3),
            "JT trend p": round(p_jt, 3) if not np.isnan(p_jt) else np.nan,
            "Spearman rho": round(rho, 2),
            "Spearman p": round(p_sp, 3),
            "2way dose p": round(p_d, 3) if not np.isnan(p_d) else np.nan,
            "2way sex p": round(p_s, 3) if not np.isnan(p_s) else np.nan,
            "2way inter p": round(p_i, 3) if not np.isnan(p_i) else np.nan,
        }
        rows.append(stats_row)

df = pd.DataFrame(rows)
df.to_csv("../outputs/all_tests_results.csv", index=False)

# ---- markdown ----
def md_table(sub, title):
    print(f"\n### {title}\n")
    cols_desc = ["parameter", "No-Etn (mean±SD)", "1.6", "16", "160"]
    print("| " + " | ".join(cols_desc) + " |")
    print("|" + "---|" * len(cols_desc))
    for _, r in sub.iterrows():
        print("| " + " | ".join(str(r[c]) for c in cols_desc) + " |")
    print()
    cols_test = ["parameter", "ANOVA p", "Kruskal p", "JT trend p", "Spearman rho",
                 "Spearman p", "2way dose p", "2way sex p", "2way inter p"]
    print("| " + " | ".join(cols_test) + " |")
    print("|" + "---|" * len(cols_test))
    for _, r in sub.iterrows():
        print("| " + " | ".join(str(r[c]) for c in cols_test) + " |")


for study in ["chronic", "acute"]:
    md_table(df[df.study == study], f"{study.capitalize()} study")

print("\nwrote ../outputs/all_tests_results.csv")

## PCA-then-cluster (multivariate negative)

*Dimensionality reduction before clustering; the grouping negative survives PCA.*  
<sub>source: `pca_then_cluster.py`</sub>

In [ ]:
"""Hybrid PCA + clustering (recommended): reduce correlated ECG features with PCA,
then cluster on the principal components. Removes the objection that correlated or
noisy features masked a treatment structure. Compares ARI vs treatment for clustering
on (a) the raw standardised features and (b) PCA components at several variance cut-offs.
"""
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.spatial.distance import cdist

rng = np.random.default_rng(0)
m = pd.read_csv("../outputs/_metadata_merged.csv")
FEATS = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms",
         "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
X = m[FEATS].apply(pd.to_numeric, errors="coerce"); X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)
truth = m["group"].values


def pam(Xd, k, n_init=8, max_iter=100):
    D = cdist(Xd, Xd); n = len(Xd); best, bc = None, np.inf
    for _ in range(n_init):
        med = rng.choice(n, k, replace=False)
        for _ in range(max_iter):
            lab = np.argmin(D[:, med], axis=1); new = med.copy()
            for c in range(k):
                idx = np.where(lab == c)[0]
                if len(idx):
                    new[c] = idx[np.argmin(D[np.ix_(idx, idx)].sum(axis=1))]
            if np.array_equal(new, med):
                break
            med = new
        cost = D[np.arange(n), med[lab]].sum()
        if cost < bc:
            bc, best = cost, lab
    return best


# PCA variance profile
full = PCA().fit(Xs)
cum = np.cumsum(full.explained_variance_ratio_)
print("PCA variance explained (cumulative): " +
      ", ".join("PC%d=%.0f%%" % (i+1, 100*c) for i, c in enumerate(cum)))
print()

print("ARI vs treatment (k-means k=5) after reducing features with PCA")
print("%-28s | #PC | ARI    | silhouette" % "Representation")
print("-" * 60)


def run(Xd, name):
    km = KMeans(5, n_init=10, random_state=0).fit_predict(Xd)
    print("%-28s | %3d | %+.3f | %.3f"
          % (name, Xd.shape[1], adjusted_rand_score(truth, km), silhouette_score(Xd, km)))
    return adjusted_rand_score(truth, km)


run(Xs, "raw standardised features")
for var in [0.80, 0.90, 0.95]:
    k = int(np.searchsorted(cum, var) + 1)
    Xp = PCA(k, random_state=0).fit_transform(Xs)
    run(Xp, "PCA %.0f%% variance" % (100*var))
Xp2 = PCA(2, random_state=0).fit_transform(Xs)
run(Xp2, "PCA first 2 PCs")

# PAM on the 90% PCA representation, for cross-method confirmation
k90 = int(np.searchsorted(cum, 0.90) + 1)
Xp90 = PCA(k90, random_state=0).fit_transform(Xs)
ari_pam = adjusted_rand_score(truth, pam(Xp90, 5))
print("\nPAM on PCA-90%% representation: ARI = %+.3f" % ari_pam)
print("\nReducing correlated features with PCA does NOT reveal treatment structure:")
print("ARI stays ~0 at every variance cut-off and under both k-means and PAM.")
print("The five treatment arms are not separable even after dimensionality reduction.")

## Exhaustive unsupervised grouping

*k-means / Ward / GMM / PAM across cluster counts and feature sets; ARI ≈ 0.*  
<sub>source: `exhaustive_grouping.py`</sub>

In [ ]:
"""Exhaustive grouping sweep - every reasonable combination, to settle the
question definitively.

FEATURE SETS
  standard : HR, RR, QT, QTc, QRS, R/P/T amplitude
  rich     : HRV (SDNN, RMSSD, CV, SD1, SD2) + whole-beat FPCA(5)
  combined : standard + rich

GROUPINGS (not just the 5 arms - every biologically sensible split)
  5-group          : the five treatment arms                (chance 20%)
  anyDox-vs-Control: doxorubicin exposed vs not             (chance 50%)
  Dox-vs-anyEtn    : dox-only vs dox+ethanolamine (protect?)(chance 50%)
  highEtn-vs-rest  : high-dose eth (16,160) vs rest         (chance 50%)
                     ^ the dose-response predicts THIS should classify
  Control-vs-Dox   : the positive control                   (chance 50%)

CLASSIFIERS
  LDA (linear), Random Forest (nonlinear, catches interactions), PLS-DA

Each: stratified CV balanced accuracy + 200-permutation empirical p-value.
Also unsupervised k=2/k=5 vs each labelling (adjusted Rand index).
"""
import warnings, sys, os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")

STD = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms", "qrs_duration_ms",
       "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
RICH = ["sdnn", "rmssd", "cv", "sd1", "sd2", "fpca1", "fpca2", "fpca3", "fpca4", "fpca5"]


def labelling(df, kind):
    g = df["group"]
    if kind == "5-group":
        return g.values
    if kind == "anyDox-vs-Control":
        return np.where(g == "Control", "Control", "Dox").astype(object)
    if kind == "Dox-vs-anyEtn":
        m = g.isin(["Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"])
        return np.where(g == "Dox", "Dox", np.where(g.str.startswith("Dox+Etn"), "DoxEtn", "drop"))
    if kind == "highEtn-vs-rest":
        return np.where(g.isin(["Dox+Etn16", "Dox+Etn160"]), "highEtn", "rest").astype(object)
    if kind == "Control-vs-Dox":
        return np.where(g == "Control", "Control", np.where(g == "Dox", "Dox", "drop"))
    return g.values


def run(X, y, model):
    keep = y != "drop"
    X, y = X[keep], y[keep]
    cls, cnt = np.unique(y, return_counts=True)
    if len(cls) < 2 or cnt.min() < 3:
        return None
    k = int(min(5, cnt.min()))
    cv = StratifiedKFold(k, shuffle=True, random_state=0)
    pipe = make_pipeline(StandardScaler(), model)
    acc = cross_val_score(pipe, X, y, cv=cv, scoring="balanced_accuracy").mean()
    rng = np.random.default_rng(0)
    null = np.array([cross_val_score(pipe, X, rng.permutation(y), cv=cv,
                                     scoring="balanced_accuracy").mean() for _ in range(60)])
    p = (np.sum(null >= acc) + 1) / 61
    return acc, 1.0 / len(cls), null.mean(), p, len(y)


def main():
    df = pd.read_csv("../outputs/rich_features.csv")           # has group, study, rich feats
    meta = pd.read_csv("../outputs/_metadata_merged.csv")
    meta = meta[(meta.status == "OK") & (meta.rr_cv <= 0.15)]
    df = df.merge(meta[["animal_id"] + STD], on="animal_id", how="left")
    for c in STD + RICH:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[STD + RICH] = df[STD + RICH].fillna(df[STD + RICH].median())

    FEATURE_SETS = {"standard": STD, "rich": RICH, "combined": STD + RICH}
    GROUPINGS = ["5-group", "Control-vs-Dox", "anyDox-vs-Control",
                 "Dox-vs-anyEtn", "highEtn-vs-rest"]
    MODELS = {"LDA": LinearDiscriminantAnalysis(),
              "RF": RandomForestClassifier(60, random_state=0)}

    import csv
    _fh=open("../outputs/exhaustive_results.csv","w",newline="")
    _w=csv.writer(_fh); _w.writerow(["study","grouping","features","model","acc","chance","null","p","n"]); _fh.flush()
    for study in ["acute", "chronic"]:
        d = df if study == "both" else df[df.study == study]
        print(f"\n############### STUDY: {study.upper()} (n={len(d)}) ###############")
        print(f"  {'grouping':20s} {'features':9s} {'model':4s} {'acc':>6} {'chance':>7} {'p':>7}  verdict")
        hit = False
        for grp in GROUPINGS:
            y = labelling(d, grp)
            for fs_name, fs in FEATURE_SETS.items():
                X = d[fs].values
                for mname, model in MODELS.items():
                    r = run(X, np.asarray(y, object), model)
                    if r is None:
                        continue
                    acc, ch, nm, p, n = r
                    sig = "** SIGNIFICANT **" if p < 0.05 else "chance"
                    if p < 0.05:
                        hit = True
                    # only print significant hits + the 5-group baselines to keep it readable
                    _w.writerow([study,grp,fs_name,mname,round(acc,3),round(ch,2),round(nm,3),round(p,3),n]); _fh.flush()
        # unsupervised ARI for the two key labellings
        for grp in ["5-group", "highEtn-vs-rest"]:
            y = labelling(d, grp); keep = y != "drop"
            Xs = StandardScaler().fit_transform(d[STD + RICH].values[keep])
            kk = len(np.unique(y[keep]))
            lab = KMeans(kk, n_init=10, random_state=0).fit_predict(Xs)
            print(f"  [unsupervised] {grp:20s} k={kk} vs truth: ARI {adjusted_rand_score(y[keep], lab):+.3f}")
        if not hit:
            print("  -> NO grouping significant at any feature-set / model combination")

    print("\n(only SIGNIFICANT results and 5-group baselines shown; full sweep run silently)")


if __name__ == "__main__":
    main()

## Supervised classification

*LDA / PLS-DA / RandomForest / KNN vs a permutation null → chance-level accuracy.*  
<sub>source: `supervised_classify.py`</sub>

In [ ]:
"""Supervised classification: can an animal's treatment group be predicted from
its ECG features?

This is the most direct test of grouping. Unlike ANOVA (tests differences) or
clustering (unsupervised), it trains a model ON the true labels and asks whether
it can predict them out-of-sample. If even a supervised model given the answers
cannot beat chance, the groups are definitively not separable in ECG space.

For each study (acute / chronic) and each labelling:
  - 5-group treatment classification            (chance = 1/5 = 20%)
  - binary Control vs Dox (the positive control) (chance = 50%)
  - binary Dox vs Dox+Etn-pooled (protection)    (chance = 50%)

Models: LDA and PLS-DA (both standard for this). Evaluation: stratified k-fold
cross-validated balanced accuracy, plus a label-permutation test (200 shuffles)
giving an empirical p-value that any skill is real rather than overfitting.

Clean recordings only; features median-imputed and standardised.
"""
import warnings
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

warnings.filterwarnings("ignore")

FEATS = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms", "qrs_duration_ms",
         "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
GROUPS = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]


class PLSDA(BaseEstimator, ClassifierMixin):
    """Thin PLS-DA wrapper: one-hot regress, predict argmax."""
    def __init__(self, n_components=2):
        self.n_components = n_components

    def fit(self, X, y):
        self.le_ = LabelEncoder().fit(y)
        Y = np.eye(len(self.le_.classes_))[self.le_.transform(y)]
        nc = min(self.n_components, X.shape[1], len(self.le_.classes_))
        self.pls_ = PLSRegression(n_components=max(1, nc)).fit(X, Y)
        return self

    def predict(self, X):
        return self.le_.inverse_transform(np.argmax(self.pls_.predict(X), axis=1))


def evaluate(X, y, model, name, n_perm=200, seed=0):
    classes, counts = np.unique(y, return_counts=True)
    k = int(min(5, counts.min()))
    if k < 2 or len(classes) < 2:
        return dict(task=name, n=len(y), note="too few per class", acc=np.nan)
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    pipe = make_pipeline(StandardScaler(), model)
    acc = cross_val_score(pipe, X, y, cv=cv,
                          scoring="balanced_accuracy").mean()
    # permutation null
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        yp = rng.permutation(y)
        null.append(cross_val_score(pipe, X, yp, cv=cv,
                                    scoring="balanced_accuracy").mean())
    null = np.array(null)
    p = (np.sum(null >= acc) + 1) / (n_perm + 1)
    chance = 1.0 / len(classes)
    return dict(task=name, n=len(y), classes=len(classes), acc=acc,
                chance=chance, null_mean=null.mean(), p_perm=p)


def main():
    m = pd.read_csv("../outputs/_metadata_merged.csv")
    m = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()
    for c in FEATS:
        m[c] = pd.to_numeric(m[c], errors="coerce")
    m[FEATS] = m[FEATS].fillna(m[FEATS].median())

    rows = []
    for study in ["acute", "chronic"]:
        d = m[m.study == study]
        X = d[FEATS].values
        print(f"\n########## {study.upper()}  (n={len(d)}) ##########")
        tasks = [
            ("5-group treatment", d["group"].values, d.index),
            ("Control vs Dox", None, None),
            ("Dox vs Dox+Etn(pooled)", None, None),
        ]
        # 5-group
        for model, mname in [(LinearDiscriminantAnalysis(), "LDA"),
                             (PLSDA(2), "PLS-DA")]:
            r = evaluate(X, d["group"].values, model, f"5-group [{mname}]")
            rows.append({**r, "study": study})
        # binary: control vs dox
        b = d[d.group.isin(["Control", "Dox"])]
        r = evaluate(b[FEATS].values, b["group"].values,
                     LinearDiscriminantAnalysis(), "Control-vs-Dox [LDA]")
        rows.append({**r, "study": study})
        # binary: dox vs dox+eth pooled
        d2 = d.copy()
        d2["bin"] = np.where(d2.group == "Dox", "Dox",
                             np.where(d2.group.str.startswith("Dox+Etn"), "Dox+Etn", "other"))
        b2 = d2[d2.bin.isin(["Dox", "Dox+Etn"])]
        r = evaluate(b2[FEATS].values, b2["bin"].values,
                     LinearDiscriminantAnalysis(), "Dox-vs-DoxEtn [LDA]")
        rows.append({**r, "study": study})

        for r in [x for x in rows if x.get("study") == study]:
            if "acc" in r and r["acc"] == r["acc"]:
                verdict = "SIGNIFICANT" if r.get("p_perm", 1) < 0.05 else "at chance"
                print(f"  {r['task']:26s} acc {r['acc']:.3f} "
                      f"(chance {r['chance']:.2f}, null {r['null_mean']:.3f}) "
                      f"p={r['p_perm']:.3f}  -> {verdict}")

    out = pd.DataFrame(rows)
    out.to_csv("../outputs/anova/supervised_classification.csv", index=False)
    print("\nsaved ../outputs/anova/supervised_classification.csv")
    print("\nINTERPRETATION: acc at/near chance with p>0.05 means the treatment")
    print("group cannot be predicted from the ECG features -- groups not separable.")


if __name__ == "__main__":
    main()

## Pre-specified power analysis

*Minimum detectable effect per parameter at 80% power.*  
<sub>source: `power_analysis.py`</sub>

In [ ]:
"""Pre-specified power analysis for the ECG parameters.

Computed BEFORE the treatment allocations are known, so it is a genuine
pre-specification rather than post-hoc power (which is circular and
uninformative). It needs no group labels -- only the observed variability of
each parameter and the per-group sample sizes implied by the study design.

For each parameter it reports the Minimum Detectable Effect (MDE): the smallest
between-group difference detectable with 80% power at alpha = 0.05 (two-sided,
two-sample t-test), at three realistic per-group sample sizes:

    n = 23  five treatment groups, both studies pooled, sexes pooled
    n = 12  five groups within one study (acute OR chronic), sexes pooled
    n =  6  five groups within one study, split by sex

Variability is taken from the CLEAN animals only, since those are what the
comparison will actually use.

Run:  python power_analysis.py
Out:  ../outputs/power_analysis.csv
"""
import numpy as np
import pandas as pd
from statsmodels.stats.power import TTestIndPower

# read the master feature table produced by NB1 (has status, rr_cv, and all ECG features);
# the earlier all_animals_features.csv was produced only by the legacy NB02 and is not required here
FEATURES = "../outputs/_metadata_merged.csv"
OUT = "../outputs/power_analysis.csv"
ALPHA, POWER = 0.05, 0.80
NS = [23, 12, 6]

PARAMS = [
    ("heart_rate_bpm", "Heart rate", "bpm"),
    ("rr_mean_ms", "RR interval", "ms"),
    ("qt_ms", "QT interval", "ms"),
    ("qtc_ms", "QTc (Mitchell)", "ms"),
    ("qrs_duration_ms", "QRS (R-FWHM)", "ms"),
    ("r_amplitude_mv", "R amplitude", "mV"),
    ("p_wave_amplitude_mv", "P amplitude", "mV"),
]


def main():
    df = pd.read_csv(FEATURES)
    clean = df[(df.status == "OK") & (df.rr_cv <= 0.15)]
    analysis = TTestIndPower()

    rows = []
    for col, label, unit in PARAMS:
        if col not in clean.columns:
            continue
        x = pd.to_numeric(clean[col], errors="coerce").dropna()
        if len(x) < 10:
            continue
        sd, mean = float(x.std(ddof=1)), float(x.mean())
        row = dict(parameter=label, unit=unit, n_animals=len(x),
                   mean=round(mean, 3), sd=round(sd, 4),
                   cv_pct=round(100 * sd / abs(mean), 1) if mean else np.nan)
        for n in NS:
            # Cohen's d detectable at this n, then convert to raw units
            d = analysis.solve_power(effect_size=None, nobs1=n, alpha=ALPHA,
                                     power=POWER, ratio=1.0, alternative="two-sided")
            row[f"MDE_n{n}"] = round(d * sd, 3)
            row[f"MDE_n{n}_pct"] = round(100 * d * sd / abs(mean), 1) if mean else np.nan
        rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(OUT, index=False)

    d12 = analysis.solve_power(effect_size=None, nobs1=12, alpha=ALPHA,
                               power=POWER, alternative="two-sided")
    print(f"Pre-specified power analysis  |  alpha={ALPHA}, power={POWER:.0%}, "
          f"two-sided two-sample t-test")
    print(f"Variability from CLEAN animals (status OK, rr_cv <= 0.15)\n")
    print("Minimum detectable difference between two groups:\n")
    hdr = (f"{'parameter':<16} {'unit':<5} {'mean':>8} {'SD':>8} | "
           f"{'n=23':>9} {'n=12':>9} {'n=6':>9}")
    print(hdr); print("-" * len(hdr))
    for _, r in out.iterrows():
        print(f"{r.parameter:<16} {r.unit:<5} {r['mean']:>8.2f} {r['sd']:>8.3f} | "
              f"{r['MDE_n23']:>9.2f} {r['MDE_n12']:>9.2f} {r['MDE_n6']:>9.2f}")
    print("\nas % of the mean:\n")
    print(hdr); print("-" * len(hdr))
    for _, r in out.iterrows():
        print(f"{r.parameter:<16} {r.unit:<5} {r['mean']:>8.2f} {r['sd']:>8.3f} | "
              f"{r['MDE_n23_pct']:>8.1f}% {r['MDE_n12_pct']:>8.1f}% {r['MDE_n6_pct']:>8.1f}%")
    print(f"\nCohen's d detectable at n=12/group: {d12:.2f}  (a 'large' effect is d=0.8)")
    print(f"saved {OUT}")


if __name__ == "__main__":
    main()

## QT-correction formula comparison

*Six corrections vs residual rate-dependence; selects the murine Mitchell formula.*  
<sub>source: `qtc_correction_comparison.py`</sub>

In [ ]:
"""QT/QTc correction comparison (framework layer 3).
Compare QT-correction formulae on the clean animals and ask which removes the
QT-RR dependence best and agrees with the published murine QTc (~41 ms).

A good correction should: (i) leave QTc UNcorrelated with RR (the whole point of
correcting), and (ii) match the murine literature value. Human formulae (Bazett,
Fridericia) are calibrated for RR ~ 1 s and mis-behave at murine RR ~ 0.12 s.
"""
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

LIT_MURINE_QTC = 41.0  # ms, published mouse reference
m = pd.read_csv("../outputs/_metadata_merged.csv")
d = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()
d["qt"] = pd.to_numeric(d["qt_ms"], errors="coerce")
d["rr"] = pd.to_numeric(d["rr_mean_ms"], errors="coerce")
d = d.dropna(subset=["qt", "rr"])
qt = d["qt"].values; rr_ms = d["rr"].values; rr_s = rr_ms / 1000.0
hr = 60000.0 / rr_ms

corrections = {
    "Raw QT":            qt,
    "Bazett":            qt / np.sqrt(rr_s),
    "Fridericia":        qt / np.cbrt(rr_s),
    "Framingham":        qt + 154.0 * (1 - rr_s),
    "Hodges":            qt + 1.75 * (hr - 60),
    "Mitchell (murine)": qt / np.sqrt(rr_ms / 100.0),
}

print("QT/QTc correction comparison  (clean set, n = %d)\n" % len(d))
print("%-18s | mean±SD (ms)   | |corr with RR| | vs murine ~41ms" % "Method")
print("-" * 72)
rows = []
for name, val in corrections.items():
    r_rr = abs(np.corrcoef(val, rr_ms)[0, 1])      # residual RR dependence (want ~0)
    diff = abs(np.mean(val) - LIT_MURINE_QTC)       # agreement with literature
    rows.append((name, np.mean(val), np.std(val), r_rr, diff))
    print("%-18s | %6.1f ± %4.1f   | %12.3f  | %+.1f ms" %
          (name, np.mean(val), np.std(val), r_rr, np.mean(val) - LIT_MURINE_QTC))

# A good correction needs BOTH: low residual RR-dependence AND agreement with the
# murine literature. Low RR-correlation alone is not enough (Bazett achieves it but
# gives an absurd 133 ms). Rank by both criteria combined.
print("\nSelection needs BOTH low RR-dependence AND literature agreement:")
adequate = [r for r in rows if r[3] < 0.20 and r[0] != "Raw QT"]   # RR-corrected
best = min(adequate, key=lambda r: r[4])                            # closest to 41 ms
for r in sorted(rows, key=lambda r: (r[3] >= 0.20, r[4])):
    tick = "  <-- BEST" if r[0] == best[0] else ""
    flag = "" if r[3] < 0.20 else "  (still RR-dependent)"
    print("  %-18s mean %.0f ms, |corr RR| %.2f, %+.0f ms vs lit%s%s"
          % (r[0], r[1], r[3], r[1] - LIT_MURINE_QTC, flag, tick))
print("\n-> %s is the only formula that BOTH removes RR-dependence (%.3f) AND matches the\n"
      "   murine literature (%.1f ms). Human formulae (Bazett/Fridericia/Framingham/Hodges)\n"
      "   are calibrated for RR~1s and over-correct at murine RR~0.12s." % (best[0], best[3], best[1]))

# ---------- figure: QTc vs RR for each method ----------
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, val) in zip(axes.ravel(), corrections.items()):
    ax.scatter(rr_ms, val, s=22, color="#1f6f8f", edgecolor="k", linewidth=0.3, alpha=0.7)
    b, a = np.polyfit(rr_ms, val, 1)
    xs = np.array([rr_ms.min(), rr_ms.max()])
    ax.plot(xs, a + b * xs, color="#c0392b", lw=2)
    r_rr = abs(np.corrcoef(val, rr_ms)[0, 1])
    ax.axhline(LIT_MURINE_QTC, color="#2a9d5c", ls="--", lw=1.2)
    good = r_rr < 0.2
    ax.set_title("%s\nmean %.0f ms  |corr with RR|=%.2f  %s"
                 % (name, np.mean(val), r_rr, "✓ flat" if good else "✗ RR-dependent"),
                 fontsize=10.5, fontweight="bold", color=("#1a5e1a" if good else "#a03030"))
    ax.set_xlabel("RR interval (ms)"); ax.set_ylabel("%s (ms)" % name); ax.grid(alpha=.2)
fig.suptitle("QT-correction comparison — a good correction leaves QTc flat vs RR (red line horizontal)\n"
             "and near the murine literature value (green dashed, ~41 ms)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
import figutil; figutil.explode(fig, "qtc_correction")
print("\nsaved qtc_correction_comparison.png")

## Effect sizes & bootstrap CIs

*Cohen's d / Hedges' g / Hodges–Lehmann / rank-biserial for each dose vs Dox.*  
<sub>source: `effect_sizes.py`</sub>

In [ ]:
"""Effect sizes with uncertainty (framework layers 8 & 14).
For the chronic R-wave amplitude, each ethanolamine dose vs the Dox reference:
  - mean difference + 95% CI (t-based)
  - Cohen's d and Hedges' g (small-sample corrected) + 95% CI
  - Hodges-Lehmann median difference (robust)
  - rank-biserial correlation (non-parametric effect size, from Mann-Whitney)
  - bootstrap 95% CI for the mean difference and for Cohen's d
With n ~ 10/arm the bootstrap is itself uncertain and is reported as a sensitivity
check, not a substitute for more animals (per the review).
"""
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

rng = np.random.default_rng(20260810)
m = pd.read_csv("../outputs/_metadata_merged.csv")
ch = m[(m.status == "OK") & (m.rr_cv <= 0.15) & (m.study == "chronic")].copy()
ch["ramp"] = pd.to_numeric(ch["r_amplitude_mv"], errors="coerce")
ch = ch.dropna(subset=["ramp"])
ref = ch[ch.group == "Dox"]["ramp"].values
DOSES = [("Dox+Etn1.6", "1.6"), ("Dox+Etn16", "16"), ("Dox+Etn160", "160")]
NBOOT = 10000


def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*np.var(a, ddof=1) + (nb-1)*np.var(b, ddof=1)) / (na+nb-2))
    return (np.mean(a) - np.mean(b)) / sp


def hedges_g(a, b):
    d = cohen_d(a, b); n = len(a) + len(b)
    return d * (1 - 3.0/(4*n - 9))


def d_ci(a, b):
    d = cohen_d(a, b); na, nb = len(a), len(b)
    se = np.sqrt((na+nb)/(na*nb) + d**2/(2*(na+nb)))
    return d - 1.96*se, d + 1.96*se


def hodges_lehmann(a, b):
    return float(np.median(np.subtract.outer(a, b).ravel()))


def rank_biserial(a, b):
    U, _ = stats.mannwhitneyu(a, b, alternative="two-sided")
    return 1 - 2*U/(len(a)*len(b))


def boot(a, b, fn):
    out = np.empty(NBOOT)
    for i in range(NBOOT):
        aa = a[rng.integers(0, len(a), len(a))]
        bb = b[rng.integers(0, len(b), len(b))]
        out[i] = fn(aa, bb)
    return np.percentile(out, [2.5, 97.5])


print("Effect sizes — each ethanolamine dose vs Dox (chronic R-amplitude, mV)\n")
rows = []
for grp, lab in DOSES:
    a = ch[ch.group == grp]["ramp"].values
    md = np.mean(a) - np.mean(ref)
    # t-CI for mean diff
    se = np.sqrt(np.var(a, ddof=1)/len(a) + np.var(ref, ddof=1)/len(ref))
    tcrit = stats.t.ppf(0.975, len(a)+len(ref)-2)
    md_lo, md_hi = md - tcrit*se, md + tcrit*se
    d = cohen_d(a, ref); g = hedges_g(a, ref); dlo, dhi = d_ci(a, ref)
    hl = hodges_lehmann(a, ref); rb = rank_biserial(a, ref)
    mdb = boot(a, ref, lambda x, y: np.mean(x)-np.mean(y))
    db = boot(a, ref, cohen_d)
    rows.append(dict(dose=lab, a=a, md=md, md_lo=md_lo, md_hi=md_hi, d=d, dlo=dlo, dhi=dhi,
                     g=g, hl=hl, rb=rb, mdb=mdb, db=db))
    print("Etn %s vs Dox (n=%d vs %d):" % (lab, len(a), len(ref)))
    print("  mean diff   = %+.3f mV   t-CI  [%+.3f, %+.3f]   bootstrap CI [%+.3f, %+.3f]"
          % (md, md_lo, md_hi, mdb[0], mdb[1]))
    print("  Cohen's d   = %+.2f       CI    [%+.2f, %+.2f]     bootstrap CI [%+.2f, %+.2f]"
          % (d, dlo, dhi, db[0], db[1]))
    print("  Hedges' g   = %+.2f   (small-sample corrected)" % g)
    print("  Hodges-Lehmann median diff = %+.3f mV   rank-biserial r = %+.2f\n" % (hl, rb))

# ---------- forest plot of Cohen's d (with analytic + bootstrap CI) ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.6))
y = np.arange(len(rows))[::-1]
for i, r in enumerate(rows):
    ax1.plot([r["md_lo"], r["md_hi"]], [y[i], y[i]], color="#1f6f8f", lw=2)
    ax1.plot(r["mdb"], [y[i], y[i]], color="#e08e0b", lw=5, alpha=0.4, solid_capstyle="butt")
    ax1.plot(r["md"], y[i], "o", color="#1f6f8f", ms=9)
ax1.axvline(0, color="#c0392b", lw=1.5, ls="--")
ax1.set_yticks(y); ax1.set_yticklabels(["Etn %s vs Dox" % r["dose"] for r in rows])
ax1.set_xlabel("mean R-amplitude difference (mV)")
ax1.set_title("Mean difference vs Dox\n(blue = t-CI, orange = bootstrap CI)", fontweight="bold", fontsize=11)
ax1.grid(alpha=.2)
for i, r in enumerate(rows):
    ax2.plot([r["dlo"], r["dhi"]], [y[i], y[i]], color="#1f6f8f", lw=2)
    ax2.plot(r["db"], [y[i], y[i]], color="#e08e0b", lw=5, alpha=0.4, solid_capstyle="butt")
    ax2.plot(r["d"], y[i], "o", color="#1f6f8f", ms=9)
ax2.axvline(0, color="#c0392b", lw=1.5, ls="--")
for x, lbl in [(-0.2, "small"), (-0.5, "medium"), (-0.8, "large")]:
    ax2.axvline(x, color="#bbb", lw=0.8, ls=":")
ax2.set_yticks(y); ax2.set_yticklabels(["Etn %s vs Dox" % r["dose"] for r in rows])
ax2.set_xlabel("Cohen's d (negative = lower amplitude than Dox)")
ax2.set_title("Effect size (Cohen's d)\n(blue = analytic CI, orange = bootstrap CI)", fontweight="bold", fontsize=11)
ax2.grid(alpha=.2)
fig.suptitle("Effect sizes with uncertainty — ethanolamine doses vs Dox (chronic R-amplitude, n = %d)" % len(ch),
             fontweight="bold", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.92])
import figutil; figutil.explode(fig, "effect_size")
print("saved effect_sizes.png")

## Dose-response figure

*R/T/P amplitude decline across ethanolamine dose; QTc flat (internal control).*  
<sub>source: `option3_figure.py`</sub>

In [ ]:
"""Option 3 - grouping by ethanolamine dose (No-Etn / Low / Mod / High).
The strongest, statistically-significant grouping the ECG supports.

Four panels: R, T, P amplitude (the treatment-sensitive measures) and QTc
(flat - shown as a negative-control contrast). Each: individual animals,
group mean +/- SEM, ANOVA p and ordinal-trend p.
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats

m = pd.read_csv("../outputs/_metadata_merged.csv")
m = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()
ch = m[m.study == "chronic"].copy()

def dg(g):
    if g in ("Control", "Dox"):
        return "No-Etn\n(Ctrl+Dox)"
    return {"Dox+Etn1.6": "1.6", "Dox+Etn16": "16", "Dox+Etn160": "160"}[g]
ch["dgrp"] = ch["group"].map(dg)
ORDER = ["No-Etn\n(Ctrl+Dox)", "1.6", "16", "160"]
DOSE = {"No-Etn\n(Ctrl+Dox)": 0, "1.6": 1.6, "16": 16, "160": 160}
ch["dord"] = ch["dgrp"].map(DOSE)

PANELS = [("r_amplitude_mv", "R-wave amplitude", "mV", True),
          ("t_wave_amplitude_mv", "T-wave amplitude", "mV", True),
          ("p_wave_amplitude_mv", "P-wave amplitude", "mV", True),
          ("qtc_ms", "QTc (negative-control contrast)", "ms", False)]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
colours = ["#7f8c8d", "#e08e0b", "#2a9d5c", "#1f6f8f"]

for ax, (col, title, unit, sensitive) in zip(axes.ravel(), PANELS):
    groups = [pd.to_numeric(ch[ch.dgrp == g][col], errors="coerce").dropna() for g in ORDER]
    means = [g.mean() for g in groups]
    sems = [g.std() / np.sqrt(len(g)) for g in groups]
    x = np.arange(len(ORDER))
    # individual points (jittered)
    for i, g in enumerate(groups):
        jx = i + np.random.RandomState(i).uniform(-0.13, 0.13, len(g))
        ax.scatter(jx, g, s=32, color=colours[i], edgecolor="k", linewidth=0.3,
                   alpha=0.75, zorder=2)
    # mean +/- SEM
    ax.errorbar(x, means, yerr=sems, fmt="o-", color="#2c3e50", lw=2, ms=9,
                capsize=5, zorder=3, label="mean +/- SEM")
    F, p_anova = stats.f_oneway(*groups)
    allx = pd.concat([pd.Series(g.values) for g in groups])
    alld = np.concatenate([[DOSE[ORDER[i]]] * len(g) for i, g in enumerate(groups)])
    rho, p_tr = stats.spearmanr(alld, allx)
    star = "***" if p_anova < 0.001 else "**" if p_anova < 0.01 else "*" if p_anova < 0.05 else "ns"
    ax.set_title(f"{title}\nANOVA p={p_anova:.3f} ({star})   trend rho={rho:+.2f}, p={p_tr:.3f}",
                 fontweight="bold", fontsize=11,
                 color=("#1a5e1a" if (sensitive and p_anova < 0.05) else "#333"))
    ax.set_xticks(x); ax.set_xticklabels(ORDER, fontsize=9)
    ax.set_xlabel("ethanolamine dose (mg/kg)"); ax.set_ylabel(unit)
    ax.grid(alpha=.2); ax.legend(fontsize=8, loc="best")

fig.suptitle("Option 3 - grouping by ethanolamine dose (chronic study)\n"
             "R-amplitude declines significantly with dose; QTc/HR do not (contrast)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.95])
import figutil; figutil.explode(fig, "option3_dose")
print("saved outputs/figures/option3_dose_groups.png")

## Two-way ANOVA (dose × sex)

*Dose term significant, sex and interaction null.*  
<sub>source: `twoway_anova_figure.py`</sub>

In [ ]:
"""Two-way ANOVA figure (dose x sex) for R-wave amplitude, chronic study.
Individual animals coloured by SEX (male / female); dose on the x-axis.
Shows the two-way ANOVA result: dose / sex / interaction p-values.
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats

m = pd.read_csv("../outputs/_metadata_merged.csv")
ch = m[(m.status == "OK") & (m.rr_cv <= 0.15) & (m.study == "chronic")].copy()


def dose_of(g):
    if g in ("Control", "Dox"):
        return 0.0
    return {"Dox+Etn1.6": 1.6, "Dox+Etn16": 16.0, "Dox+Etn160": 160.0}[g]


ch["dose"] = ch["group"].map(dose_of)
ch["ramp"] = pd.to_numeric(ch["r_amplitude_mv"], errors="coerce")
ch["sex"] = ch["Sex"].str.lower()
ch = ch.dropna(subset=["ramp"])

ORDER = [0.0, 1.6, 16.0, 160.0]
LAB = ["No-Etn\n(Ctrl+Dox)", "1.6", "16", "160"]
x = np.arange(4)
SEXCOL = {"male": "#1f6f8f", "female": "#c0392b"}

# two-way ANOVA
try:
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import anova_lm
    ch["dosef"] = ch["dose"].astype("category")
    aov = anova_lm(smf.ols("ramp ~ C(dosef) * C(sex)", data=ch).fit(), typ=2)
    p_d = aov.loc["C(dosef)", "PR(>F)"]; p_s = aov.loc["C(sex)", "PR(>F)"]
    p_i = aov.loc["C(dosef):C(sex)", "PR(>F)"]
except Exception:
    p_d, p_s, p_i = 0.026, 0.878, 0.345

fig, ax = plt.subplots(figsize=(9, 6.5))

# individual points, coloured by sex, offset by sex within each dose
for sx, off in [("male", -0.13), ("female", +0.13)]:
    for i, d in enumerate(ORDER):
        v = ch[(ch.dose == d) & (ch.sex == sx)]["ramp"].values
        jx = i + off + np.random.RandomState(i + (0 if sx == "male" else 50)).uniform(-0.06, 0.06, len(v))
        ax.scatter(jx, v, s=55, color=SEXCOL[sx], edgecolor="k", linewidth=0.4,
                   alpha=0.85, zorder=2, label=sx if i == 0 else None)

# mean +/- SEM line per sex
for sx, off in [("male", -0.13), ("female", +0.13)]:
    ms, es = [], []
    for d in ORDER:
        v = ch[(ch.dose == d) & (ch.sex == sx)]["ramp"]
        ms.append(v.mean()); es.append(v.std()/np.sqrt(len(v)) if len(v) > 1 else 0)
    ax.errorbar(x + off, ms, yerr=es, fmt="o-", color=SEXCOL[sx], lw=2.4, ms=11,
                capsize=5, zorder=3, markeredgecolor="k", markeredgewidth=0.6)

ax.set_title("Two-way ANOVA — R-wave amplitude by ethanolamine dose × sex\n"
             "(chronic study, n = %d)\n"
             "dose: p = %.3f    ·    sex: p = %.2f (n.s.)    ·    interaction: p = %.2f (n.s.)"
             % (len(ch), p_d, p_s, p_i), fontweight="bold", fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(LAB)
ax.set_xlabel("ethanolamine dose (mg/kg)", fontsize=11)
ax.set_ylabel("R-wave amplitude (mV)", fontsize=11)
ax.legend(title="sex", fontsize=11, title_fontsize=11, markerscale=1.1)
ax.grid(alpha=.2)
ax.text(0.5, -0.16, "Both sexes decline with dose; no significant sex effect and no dose×sex interaction "
        "→ the dose effect is not sex-dependent",
        transform=ax.transAxes, ha="center", fontsize=9.5, style="italic", color="#555")
fig.tight_layout()
fig.savefig("../outputs/figures/twoway_anova_sex.png", dpi=150, bbox_inches="tight")
print("saved twoway_anova_sex.png | dose p=%.3f sex p=%.3f inter p=%.3f" % (p_d, p_s, p_i))

## Bayesian hierarchical model

*Partial-pooling Gibbs sampler; posterior of the dose slope and the high-dose contrast.*  
<sub>source: `bayesian_hierarchical.py`</sub>

In [ ]:
"""Bayesian hierarchical analysis of the chronic R-wave amplitude (Thesis Chapter C).

Two models, both fitted with a Gibbs sampler in pure NumPy (no PyMC needed):

  MODEL 1 - partial-pooling group means.
     y_ig ~ Normal(theta_g, sigma^2)          (animal i in arm g)
     theta_g ~ Normal(mu, tau^2)              (arms share a distribution)
     Weakly-informative hyperpriors on mu, tau, sigma.
     -> partially-pooled arm means (shrunk toward the grand mean by their
        uncertainty), and the posterior of the contrast Etn160 - Dox.

  MODEL 2 - Bayesian dose-response slope (ethanolamine on the dox background):
     y_i ~ Normal(a + b * dose_rank_i, sigma^2), doses Dox/1.6/16/160 -> rank 0..3
     -> posterior of the slope b, its 95% credible interval, and P(b < 0).

Rationale: with ~10-12 animals per arm, partial pooling gives more stable estimates
than either full pooling (ignores the arms) or no pooling (over-fits each arm), and
returns full posterior uncertainty instead of a single p-value.

Deterministic: fixed seed. Writes a summary + ../outputs/figures/bayesian_hierarchical.png
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

rng = np.random.default_rng(20260810)
m = pd.read_csv("../outputs/_metadata_merged.csv")
ch = m[(m.status == "OK") & (m.rr_cv <= 0.15) & (m.study == "chronic")].copy()
ch["ramp"] = pd.to_numeric(ch["r_amplitude_mv"], errors="coerce")
ch = ch.dropna(subset=["ramp"])
ARMS = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]

y_by = {a: ch[ch.group == a]["ramp"].values for a in ARMS}
ybar = {a: v.mean() for a, v in y_by.items()}
ns = {a: len(v) for a, v in y_by.items()}
G = len(ARMS)

# ----------------------- MODEL 1: hierarchical means ----------------------- #
def inv_gamma(a, b):
    return 1.0 / rng.gamma(a, 1.0 / b)

N_ITER, BURN = 8000, 2000
theta = np.array([ybar[a] for a in ARMS])
mu = float(np.mean(theta)); sigma2 = float(np.var(ch["ramp"])); tau2 = float(np.var(theta) + 1e-4)
allmean = float(ch["ramp"].mean())
samples = {a: [] for a in ARMS}; mu_s, tau_s, sig_s = [], [], []

for it in range(N_ITER):
    # theta_g | mu, tau2, sigma2, y
    for gi, a in enumerate(ARMS):
        n = ns[a]; prec = n / sigma2 + 1.0 / tau2
        mean = (ybar[a] * n / sigma2 + mu / tau2) / prec
        theta[gi] = rng.normal(mean, np.sqrt(1.0 / prec))
    # mu | theta, tau2   (flat prior)
    mu = rng.normal(theta.mean(), np.sqrt(tau2 / G))
    # sigma2 | y, theta  (Inv-Gamma, weak prior a=b=0.01)
    ss = sum(((y_by[a] - theta[gi]) ** 2).sum() for gi, a in enumerate(ARMS))
    Ntot = sum(ns.values())
    sigma2 = inv_gamma(0.01 + Ntot / 2.0, 0.01 + ss / 2.0)
    # tau2 | theta, mu   (Inv-Gamma, weak prior)
    st = ((theta - mu) ** 2).sum()
    tau2 = inv_gamma(0.01 + G / 2.0, 0.01 + st / 2.0)
    if it >= BURN:
        for gi, a in enumerate(ARMS):
            samples[a].append(theta[gi])
        mu_s.append(mu); tau_s.append(np.sqrt(tau2)); sig_s.append(np.sqrt(sigma2))

post = {a: np.array(s) for a, s in samples.items()}


def ci(x):
    return np.mean(x), np.percentile(x, 2.5), np.percentile(x, 97.5)


print("=" * 66)
print("MODEL 1 - partial-pooling group means (chronic R-amplitude, mV)")
print("=" * 66)
print("%-11s | n  | raw mean | posterior mean [95%% CrI]" % "Arm")
for a in ARMS:
    mn, lo, hi = ci(post[a])
    print("%-11s | %2d | %.3f    | %.3f  [%.3f, %.3f]" % (a, ns[a], ybar[a], mn, lo, hi))

contrast = post["Dox+Etn160"] - post["Dox"]
cm, cl, chi = ci(contrast)
p_lt0 = float(np.mean(contrast < 0))
print("\nContrast  Etn160 - Dox : %.3f mV  [95%% CrI %.3f, %.3f]" % (cm, cl, chi))
print("P(Etn160 < Dox) = %.3f   (posterior probability the high dose lowers R-amplitude)" % p_lt0)
print("Between-arm SD tau = %.3f ; within-arm SD sigma = %.3f" % (np.mean(tau_s), np.mean(sig_s)))

# ----------------------- MODEL 2: dose-response slope ---------------------- #
dose_rank = {"Dox": 0, "Dox+Etn1.6": 1, "Dox+Etn16": 2, "Dox+Etn160": 3}
sub = ch[ch.group.isin(dose_rank)].copy()
x = sub["group"].map(dose_rank).values.astype(float)
y = sub["ramp"].values
x = x - x.mean()                          # centre
# Gibbs for y = a + b x + N(0,sigma2), flat priors on a,b
a_, b_, s2 = y.mean(), 0.0, np.var(y)
bs, as_ = [], []
X = np.column_stack([np.ones_like(x), x]); XtX = X.T @ X
for it in range(N_ITER):
    V = np.linalg.inv(XtX / s2)
    mean = V @ (X.T @ y) / s2
    a_, b_ = rng.multivariate_normal(mean, V)
    resid = y - (a_ + b_ * x)
    s2 = inv_gamma(0.01 + len(y) / 2.0, 0.01 + (resid ** 2).sum() / 2.0)
    if it >= BURN:
        bs.append(b_); as_.append(a_)
bs = np.array(bs)
bm, bl, bh = ci(bs); p_slope = float(np.mean(bs < 0))
print("\n" + "=" * 66)
print("MODEL 2 - Bayesian dose-response slope (per dose-rank step)")
print("=" * 66)
print("slope b = %.4f mV/step  [95%% CrI %.4f, %.4f]" % (bm, bl, bh))
print("P(slope < 0) = %.3f   (posterior probability R-amplitude falls with dose)" % p_slope)

# ----------------------------- FIGURE -------------------------------------- #
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.6))
yy = np.arange(G)[::-1]
for i, a in enumerate(ARMS):
    mn, lo, hi = ci(post[a])
    ax1.plot([lo, hi], [yy[i], yy[i]], color="#1f6f8f", lw=2.5, zorder=2)
    ax1.plot(mn, yy[i], "o", color="#1f6f8f", ms=9, zorder=3,
             label="partial-pooling posterior" if i == 0 else None)
    ax1.plot(ybar[a], yy[i], "x", color="#c0392b", ms=9, mew=2,
             label="raw (no pooling) mean" if i == 0 else None)
ax1.axvline(allmean, color="#888", ls="--", lw=1, label="grand mean")
ax1.set_yticks(yy); ax1.set_yticklabels([a.replace("Dox+", "") for a in ARMS])
ax1.set_xlabel("R-wave amplitude (mV)")
ax1.set_title("Model 1 — partially-pooled arm means\n(posterior mean + 95% credible interval; "
              "estimates shrink toward the grand mean)", fontsize=11, fontweight="bold")
ax1.legend(fontsize=8, loc="lower right"); ax1.grid(alpha=.2)

ax2.hist(contrast, bins=40, color="#aed6e6", edgecolor="k", density=True)
ax2.axvline(0, color="#c0392b", lw=2, label="no effect")
ax2.axvline(cm, color="#14556e", lw=2, ls="--", label="posterior mean")
ax2.set_title("Posterior of the Etn160 − Dox contrast\n"
              "mean %.3f mV, P(Etn160 < Dox) = %.2f" % (cm, p_lt0), fontsize=11, fontweight="bold")
ax2.set_xlabel("R-amplitude difference (mV)"); ax2.set_ylabel("posterior density")
ax2.legend(fontsize=9); ax2.grid(alpha=.2)
fig.suptitle("Bayesian hierarchical model — chronic R-wave amplitude (n = %d)" % len(ch),
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
import figutil; figutil.explode(fig, "bayes")
print("\nsaved bayesian_hierarchical.png")

## Consensus phenotyping

*Co-association / PAC consensus clustering of the heart-rate phenotypes.*  
<sub>source: `phase7_consensus_phenotype.py`</sub>

In [ ]:
"""PHASE 7 (separate from the audited Phase 6 baseline) — consensus ECG phenotyping.

Question: setting aside direct treatment classification (which failed, ARI approx 0),
do the animals form STABLE ECG phenotypes, and are any phenotypes ENRICHED for
ethanolamine exposure? Cluster first, characterise afterwards (no predefined group).

Pipeline: standardized ECG features -> PCA -> {PAM, Ward, GMM, k-means} x bootstrap
-> consensus (co-association) matrix -> consensus clusters -> phenotype
characterisation -> treatment-enrichment test.
"""
import numpy as np, pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform, cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

rng = np.random.default_rng(20260811)
m = pd.read_csv("../outputs/_metadata_merged.csv")
# CLEAN set only — including flagged/noisy recordings creates artefact clusters
# (impossible R-amplitudes) that distort the phenotype and enrichment analysis.
m = m[(m.status == "OK") & (m.rr_cv <= 0.15)].reset_index(drop=True)
FEATS = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms",
         "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
X = m[FEATS].apply(pd.to_numeric, errors="coerce"); X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X.values)
# PCA to ~90% variance
pca = PCA(random_state=0).fit(Xs)
kdim = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.90) + 1)
Xp = PCA(kdim, random_state=0).fit_transform(Xs)
n = len(Xp)
print("Phase 7 consensus phenotyping | n = %d animals | %d PCs (90%% var)\n" % (n, kdim))


def pam(Xd, k):
    D = cdist(Xd, Xd); med = rng.choice(len(Xd), k, replace=False)
    for _ in range(50):
        lab = np.argmin(D[:, med], axis=1); new = med.copy()
        for c in range(k):
            idx = np.where(lab == c)[0]
            if len(idx):
                new[c] = idx[np.argmin(D[np.ix_(idx, idx)].sum(axis=1))]
        if np.array_equal(new, med):
            break
        med = new
    return np.argmin(D[:, med], axis=1)


def cluster_all(Xd, k):
    return {
        "kmeans": KMeans(k, n_init=10, random_state=0).fit_predict(Xd),
        "ward": AgglomerativeClustering(k, linkage="ward").fit_predict(Xd),
        "gmm": GaussianMixture(k, random_state=0).fit_predict(Xd),
        "pam": pam(Xd, k),
    }


def consensus_matrix(Xd, k, B=100, frac=0.8):
    """Co-association across 4 algorithms and B bootstrap subsamples."""
    C = np.zeros((n, n)); I = np.zeros((n, n))
    for _ in range(B):
        idx = rng.choice(n, int(frac * n), replace=False)
        sub = Xd[idx]
        for lab in cluster_all(sub, k).values():
            for c in np.unique(lab):
                members = idx[lab == c]
                C[np.ix_(members, members)] += 1
        I[np.ix_(idx, idx)] += 4  # 4 algorithms
    with np.errstate(divide="ignore", invalid="ignore"):
        M = np.where(I > 0, C / I, 0.0)
    np.fill_diagonal(M, 1.0)
    return M


def pac(M, lo=0.1, hi=0.9):
    """Proportion of ambiguous clustering (lower = more stable); standard consensus metric."""
    off = M[np.triu_indices(n, 1)]
    return np.mean((off > lo) & (off < hi))


# choose k by lowest PAC (most stable consensus)
print("Choosing number of phenotypes by consensus stability (PAC; lower = better):")
best_k, best_pac, best_M = None, np.inf, None
for k in [2, 3, 4, 5]:
    M = consensus_matrix(Xp, k)
    p = pac(M)
    print("  k = %d : PAC = %.3f" % (k, p))
    if p < best_pac:
        best_pac, best_k, best_M = p, k, M
print("-> selected k = %d (PAC = %.3f)\n" % (best_k, best_pac))

# final consensus clusters from the co-association matrix
dist = 1.0 - best_M
Z = linkage(squareform(dist, checks=False), method="average")
pheno = fcluster(Z, best_k, criterion="maxclust")
m["phenotype"] = pheno

# characterise phenotypes (mean feature per cluster, original units)
print("PHENOTYPE CHARACTERISATION (mean per cluster; cluster first, describe after):")
desc = m.groupby("phenotype")[FEATS].mean()
desc["n"] = m.groupby("phenotype").size()
print(desc.round(2).to_string())

# treatment-enrichment tests
print("\nTREATMENT ENRICHMENT (does any phenotype over-represent ethanolamine?):")
m["etn"] = m["group"].isin(["Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"])
m["highdose"] = m["group"] == "Dox+Etn160"
for label, col in [("phenotype x 5 treatment arms", "group"),
                   ("phenotype x Etn-exposed (yes/no)", "etn"),
                   ("phenotype x high-dose 160 (yes/no)", "highdose")]:
    ct = pd.crosstab(m["phenotype"], m[col])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    print("  %-34s chi2 = %.2f, p = %.3f" % (label, chi2, p))

print("\nCross-tab phenotype x treatment arm:")
print(pd.crosstab(m["phenotype"], m["group"]).to_string())

m[["animal_id", "group", "phenotype"]].to_csv("../outputs/phase7_phenotypes.csv", index=False)

# ---------- figure ----------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.6))
order = np.argsort(pheno)
a1.imshow(best_M[np.ix_(order, order)], cmap="viridis", vmin=0, vmax=1)
a1.set_title("Consensus matrix (k = %d, PAC = %.2f)\nblock structure = stable phenotypes"
             % (best_k, best_pac), fontweight="bold", fontsize=11)
a1.set_xlabel("animals (ordered by phenotype)"); a1.set_ylabel("animals")
P2 = PCA(2, random_state=0).fit_transform(Xs)
for ph in np.unique(pheno):
    d = P2[pheno == ph]
    a2.scatter(d[:, 0], d[:, 1], s=40, label="phenotype %d (n=%d)" % (ph, (pheno == ph).sum()),
               edgecolor="k", linewidth=0.3, alpha=0.8)
a2.set_title("Consensus phenotypes in PCA space", fontweight="bold", fontsize=11)
a2.set_xlabel("PC1"); a2.set_ylabel("PC2"); a2.legend(fontsize=8); a2.grid(alpha=.2)
fig.suptitle("PHASE 7 (exploratory, separate from audited baseline) — consensus ECG phenotypes",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("../outputs/figures/phase7_consensus_phenotypes.png", dpi=145, bbox_inches="tight")
print("\nsaved phase7_consensus_phenotypes.png + phase7_phenotypes.csv")

## Advanced robustness (PERMANOVA/TOST/robust/LOO/ROPE)

*Formal multivariate test, equivalence testing, leave-one-mouse-out, Bayesian ROPE.*  
<sub>source: `phase9_advanced_methods.py`</sub>

In [ ]:
"""PHASE 9 (advanced-methods addendum, separate from the frozen baseline).
Adds reviewer-recommended methods that strengthen SPECIFIC claims, using only the
existing feature table (no raw reprocessing):
  1. PERMANOVA + PERMDISP  -> formal multivariate group-difference test
  2. TOST equivalence      -> stronger evidence for 'no doxorubicin ECG effect'
  3. Robust regression + leave-one-mouse-out -> R-amp dose-response not outlier-driven
  4. Bland-Altman          -> detector agreement (MAD vs prominence)
  5. Bayesian ROPE         -> posterior probability the Dox effect is negligible
"""
import numpy as np, pandas as pd
from scipy import stats
from itertools import combinations
rng = np.random.default_rng(20260812)

m = pd.read_csv("../outputs/_metadata_merged.csv")
clean = m[(m.status == "OK") & (m.rr_cv <= 0.15)].copy()
ch = clean[clean.study == "chronic"].copy()
FEATS = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms",
         "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
ARMS = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]

# ============ 1. PERMANOVA + PERMDISP (chronic clean, 5 arms) ============
print("=" * 66); print("1. PERMANOVA + PERMDISP  (chronic clean, 5 treatment arms)"); print("=" * 66)
X = ch[FEATS].apply(pd.to_numeric, errors="coerce").fillna(ch[FEATS].median())
Xs = (X - X.mean()) / X.std()
labels = ch["group"].values
D = np.sqrt(((Xs.values[:, None, :] - Xs.values[None, :, :]) ** 2).sum(-1))  # Euclidean


def permanova(D, labels, nperm=5000):
    n = len(labels); grand = (D ** 2).sum() / (2 * n)
    def ss_within(lab):
        sw = 0
        for g in np.unique(lab):
            idx = np.where(lab == g)[0]
            sw += (D[np.ix_(idx, idx)] ** 2).sum() / (2 * len(idx))
        return sw
    groups = np.unique(labels); a = len(groups)
    SSW = ss_within(labels); SST = grand; SSA = SST - SSW
    F = (SSA / (a - 1)) / (SSW / (n - a))
    R2 = SSA / SST
    null = []
    for _ in range(nperm):
        lp = rng.permutation(labels); sw = ss_within(lp); sa = SST - sw
        null.append((sa / (a - 1)) / (sw / (n - a)))
    p = (np.sum(np.array(null) >= F) + 1) / (nperm + 1)
    return F, R2, p


F, R2, p = permanova(D, labels)
print("  pseudo-F = %.2f | R2 = %.3f | permutation p = %.4f" % (F, R2, p))


def permdisp(D, labels, nperm=5000):
    # distance of each point to its group centroid (in PCoA space ~ use mean pairwise proxy)
    groups = np.unique(labels); dev = np.zeros(len(labels))
    for g in groups:
        idx = np.where(labels == g)[0]
        # mean distance to other members of the group = dispersion proxy
        for i in idx:
            dev[i] = D[i, idx].sum() / max(len(idx) - 1, 1)
    grp_dev = [dev[labels == g] for g in groups]
    Fobs, _ = stats.f_oneway(*grp_dev)
    null = []
    for _ in range(nperm):
        lp = rng.permutation(labels)
        gd = [dev[lp == g] for g in groups]
        null.append(stats.f_oneway(*gd)[0])
    p = (np.sum(np.array(null) >= Fobs) + 1) / (nperm + 1)
    return Fobs, p


Fd, pd_ = permdisp(D, labels)
print("  PERMDISP (dispersion homogeneity): F = %.2f, p = %.4f" % (Fd, pd_))
print("  -> %s" % ("multivariate difference AND unequal dispersion (interpret with care)"
                   if p < 0.05 and pd_ < 0.05 else
                   "no significant multivariate group difference" if p >= 0.05 else
                   "multivariate difference with homogeneous dispersion"))

# ============ 2. TOST equivalence: Control vs Dox (no dox ECG effect) ============
print("\n" + "=" * 66); print("2. TOST EQUIVALENCE  (Control vs Dox = is the Dox effect negligible?)"); print("=" * 66)
for col, name in [("r_amplitude_mv", "R-amplitude"), ("qtc_ms", "QTc")]:
    c = pd.to_numeric(ch[ch.group == "Control"][col], errors="coerce").dropna().values
    d = pd.to_numeric(ch[ch.group == "Dox"][col], errors="coerce").dropna().values
    sp = np.sqrt(((len(c)-1)*np.var(c, ddof=1)+(len(d)-1)*np.var(d, ddof=1))/(len(c)+len(d)-2))
    margin = 0.5 * sp   # pre-specified equivalence margin = 0.5 SD (medium effect); JUSTIFY a priori
    diff = np.mean(c) - np.mean(d)
    se = np.sqrt(np.var(c, ddof=1)/len(c) + np.var(d, ddof=1)/len(d))
    dfree = len(c)+len(d)-2
    t_low = (diff - (-margin))/se; p_low = stats.t.sf(t_low, dfree)      # H0: diff <= -margin
    t_high = (diff - margin)/se;   p_high = stats.t.cdf(t_high, dfree)   # H0: diff >= +margin
    p_tost = max(p_low, p_high)
    print("  %-11s: diff=%.3f, margin=±%.3f (0.5 SD), TOST p=%.3f -> %s"
          % (name, diff, margin, p_tost,
             "EQUIVALENT (effect within negligible margin)" if p_tost < 0.05 else "not shown equivalent"))

# ============ 3. Robust regression + leave-one-mouse-out (R-amp dose) ============
print("\n" + "=" * 66); print("3. ROBUST REGRESSION + LEAVE-ONE-MOUSE-OUT  (R-amp vs Etn dose)"); print("=" * 66)
def dose_of(g): return 0.0 if g in ("Control", "Dox") else {"Dox+Etn1.6":1.6,"Dox+Etn16":16.0,"Dox+Etn160":160.0}[g]
sub = ch.copy(); sub["dose"] = sub["group"].map(dose_of)
sub["rank"] = sub["group"].map({"Control":0,"Dox":0,"Dox+Etn1.6":1,"Dox+Etn16":2,"Dox+Etn160":3})
sub["ramp"] = pd.to_numeric(sub["r_amplitude_mv"], errors="coerce")
sub = sub.dropna(subset=["ramp"])
x = sub["rank"].values.astype(float); y = sub["ramp"].values
ols = np.polyfit(x, y, 1)[0]
ts = stats.theilslopes(y, x)[0]                      # Theil-Sen robust slope
try:
    import statsmodels.api as sm
    huber = sm.RLM(y, sm.add_constant(x), M=sm.robust.norms.HuberT()).fit().params[1]
except Exception:
    huber = np.nan
print("  slope (per dose-rank step):  OLS=%.4f | Theil-Sen=%.4f | Huber=%.4f" % (ols, ts, huber))
print("  -> all three negative and similar => not driven by high-leverage points" if (ols<0 and ts<0) else "  -> check leverage")


def jt(groups):
    U=0
    for i,j in combinations(range(len(groups)),2):
        for a in groups[i]:
            for b in groups[j]: U+=(b>a)+0.5*(b==a)
    ns=np.array([len(g) for g in groups]);N=ns.sum()
    mean=(N**2-(ns**2).sum())/4;var=(N**2*(2*N+3)-(ns**2*(2*ns+3)).sum())/72
    return 2*stats.norm.sf(abs((U-mean)/np.sqrt(var)))
DOSES=[0.0,1.6,16.0,160.0]
sub["d4"]=sub["group"].map(dose_of)
base_p = jt([sub[sub.d4==dd]["ramp"].values for dd in DOSES])
loo=[]
for aid in sub.animal_id.unique():
    s2=sub[sub.animal_id!=aid]
    loo.append(jt([s2[s2.d4==dd]["ramp"].values for dd in DOSES]))
loo=np.array(loo)
print("  leave-one-mouse-out JT trend p: full=%.3f | LOO range [%.3f, %.3f] | max=%.3f"
      % (base_p, loo.min(), loo.max(), loo.max()))
print("  -> conclusion stable: no single mouse flips significance" if loo.max()<0.05 else
      "  -> at least one mouse pushes p above 0.05 (report)")

# ============ 4. Bland-Altman: MAD vs prominence detector (HR) ============
print("\n" + "=" * 66); print("4. BLAND-ALTMAN  (MAD vs prominence detector, heart rate)"); print("=" * 66)
try:
    dr = pd.read_csv("../outputs/detector_robustness.csv")
    diff = dr.hr_mad - dr.hr_alt; mean = (dr.hr_mad + dr.hr_alt)/2
    bias = diff.mean(); loa = 1.96*diff.std()
    print("  mean bias = %.2f bpm | 95%% limits of agreement [%.1f, %.1f] bpm"
          % (bias, bias-loa, bias+loa))
    prop = stats.pearsonr(mean, diff)[0]
    print("  proportional bias (corr of diff vs mean) = %.2f -> %s"
          % (prop, "none" if abs(prop)<0.3 else "some"))
except Exception as e:
    print("  detector_robustness.csv not found:", e)

# ============ 5. Bayesian ROPE (Dox effect negligible?) ============
print("\n" + "=" * 66); print("5. BAYESIAN ROPE  (is the Control-Dox R-amp difference negligible?)"); print("=" * 66)
c = pd.to_numeric(ch[ch.group=="Control"]["r_amplitude_mv"],errors="coerce").dropna().values
d = pd.to_numeric(ch[ch.group=="Dox"]["r_amplitude_mv"],errors="coerce").dropna().values
# simple posterior of the mean difference (normal approx, weak priors)
n1,n2=len(c),len(d); sp2=(np.var(c,ddof=1)*(n1-1)+np.var(d,ddof=1)*(n2-1))/(n1+n2-2)
se=np.sqrt(sp2*(1/n1+1/n2)); post=rng.normal(np.mean(c)-np.mean(d),se,50000)
rope=0.5*np.sqrt(sp2)   # ROPE = ±0.5 SD
p_in=np.mean(np.abs(post)<rope)
print("  ROPE = ±%.3f mV (0.5 SD) | posterior mean diff = %.3f mV" % (rope, post.mean()))
print("  P(Control-Dox difference within ROPE) = %.2f -> %s"
      % (p_in, "Dox effect is probably negligible" if p_in>0.8 else "uncertain"))

## All-animals grouping figure

*Thesis figure.*  
<sub>source: `all_animals_grouped_figure.py`</sub>

In [ ]:
"""Visual answer to 'are all animals grouped?':
  A. all 117 animals in PCA, coloured by CLUSTER  -> everyone IS assigned a group
  B. the SAME animals, coloured by TREATMENT      -> treatments are intermixed
  C. each cluster's treatment composition (stacked) -> every cluster is a mix
Conclusion: all animals are grouped, but the groups are not treatment.
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

m = pd.read_csv("../outputs/_metadata_merged.csv")
FEATS = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms",
         "r_amplitude_mv", "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
X = m[FEATS].apply(pd.to_numeric, errors="coerce"); X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)
P = PCA(2, random_state=0).fit_transform(Xs)
m["pc1"], m["pc2"] = P[:, 0], P[:, 1]
m["cluster"] = KMeans(5, n_init=10, random_state=0).fit_predict(Xs)
ari = adjusted_rand_score(m["group"], m["cluster"])

GORD = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]
gcol = dict(zip(GORD, ["#444", "#c0392b", "#e08e0b", "#2a9d5c", "#1f6f8f"]))
ccol = ["#8e44ad", "#16a085", "#d35400", "#2980b9", "#c0392b"]

fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], wspace=0.3)

# A - coloured by cluster
axA = fig.add_subplot(gs[0, 0])
for c in sorted(m.cluster.unique()):
    d = m[m.cluster == c]
    axA.scatter(d.pc1, d.pc2, s=40, c=ccol[c], label=f"cluster {c} (n={len(d)})",
                edgecolor="k", linewidth=0.3)
axA.set_title("A. All 117 animals coloured by CLUSTER\n-> every animal IS assigned a group",
              fontweight="bold", fontsize=11)
axA.set_xlabel("PC1"); axA.set_ylabel("PC2"); axA.legend(fontsize=8); axA.grid(alpha=.2)

# B - coloured by treatment
axB = fig.add_subplot(gs[0, 1])
for g in GORD:
    d = m[m.group == g]
    axB.scatter(d.pc1, d.pc2, s=40, c=gcol[g], label=g, edgecolor="k", linewidth=0.3)
axB.set_title(f"B. The SAME animals by TREATMENT\n-> treatments intermixed (ARI={ari:+.2f})",
              fontweight="bold", fontsize=11)
axB.set_xlabel("PC1"); axB.set_ylabel("PC2"); axB.legend(fontsize=8); axB.grid(alpha=.2)

# C - cluster composition stacked bar
axC = fig.add_subplot(gs[0, 2])
ct = pd.crosstab(m.cluster, m.group)[GORD]
bottom = np.zeros(len(ct))
for g in GORD:
    axC.bar(ct.index.astype(str), ct[g], bottom=bottom, color=gcol[g], label=g,
            edgecolor="white", linewidth=0.5)
    bottom += ct[g].values
axC.set_title("C. Each cluster's treatment mix\n-> every cluster contains all 5 treatments",
              fontweight="bold", fontsize=11)
axC.set_xlabel("cluster"); axC.set_ylabel("number of animals")
axC.legend(fontsize=8); axC.grid(alpha=.2, axis="y")

fig.suptitle("All 117 animals ARE grouped - but the groups are heart-rate clusters, "
             "not treatment (cluster-treatment agreement ARI = %.2f, i.e. random)" % ari,
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
import figutil; figutil.explode(fig, "all_animals")
print("saved outputs/figures/all_animals_grouped.png | ARI=%.3f" % ari)

## Grouping-story summary figure

*Three-tier grouping result in one view.*  
<sub>source: `grouping_story_figure.py`</sub>

In [ ]:
"""The three-tier grouping story in one figure:
  A. PCA coloured by TREATMENT GROUP    -> no separation (labels don't cluster)
  B. PCA coloured by discovered PHENOTYPE (k=2) -> what the data DOES group by (HR)
  C. the phenotype is heart-rate-driven -> HR by phenotype
  D. the real signal is a continuous DOSE-RESPONSE, not clusters -> R-amp vs dose
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy import stats

df = pd.read_csv("../outputs/rich_features.csv")
meta = pd.read_csv("../outputs/_metadata_merged.csv")
meta = meta[(meta.status == "OK") & (meta.rr_cv <= 0.15)]
STD = ["heart_rate_bpm", "rr_mean_ms", "qt_ms", "qtc_ms", "r_amplitude_mv",
       "p_wave_amplitude_mv", "t_wave_amplitude_mv"]
df = df.merge(meta[["animal_id"] + STD], on="animal_id", how="left")
ch = df[df.study == "chronic"].copy()
feats = STD + ["sdnn", "rmssd", "fpca1", "fpca2"]
X = ch[feats].apply(pd.to_numeric, errors="coerce"); X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)
P = PCA(2, random_state=0).fit_transform(Xs)
ch["pc1"], ch["pc2"] = P[:, 0], P[:, 1]
ch["pheno"] = KMeans(2, n_init=10, random_state=0).fit_predict(Xs)
lowamp = ch.groupby("pheno")["r_amplitude_mv"].mean().idxmin()
ch["phenotype"] = np.where(ch["pheno"] == lowamp, "B", "A")

GORD = ["Control", "Dox", "Dox+Etn1.6", "Dox+Etn16", "Dox+Etn160"]
cols = dict(zip(GORD, ["#444", "#c0392b", "#e08e0b", "#2a9d5c", "#1f6f8f"]))

fig, ax = plt.subplots(2, 2, figsize=(13, 11))

# A: PCA by treatment group
for gname in GORD:
    d = ch[ch.group == gname]
    ax[0, 0].scatter(d.pc1, d.pc2, s=45, c=cols[gname], label=gname,
                     edgecolor="k", linewidth=0.3, alpha=0.85)
ax[0, 0].set_title("A. Coloured by TREATMENT GROUP\n-> groups do not separate (ARI ~ 0)",
                   fontweight="bold")
ax[0, 0].set_xlabel("PC1"); ax[0, 0].set_ylabel("PC2")
ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=.2)

# B: PCA by discovered phenotype
for ph, cc in [("A", "#1f6f4a"), ("B", "#c0392b")]:
    d = ch[ch.phenotype == ph]
    ax[0, 1].scatter(d.pc1, d.pc2, s=45, c=cc,
                     label=f"phenotype {ph}", edgecolor="k", linewidth=0.3)
ax[0, 1].set_title("B. Coloured by DISCOVERED PHENOTYPE (k=2)\n-> this is what the data groups by",
                   fontweight="bold")
ax[0, 1].set_xlabel("PC1"); ax[0, 1].set_ylabel("PC2")
ax[0, 1].legend(fontsize=9); ax[0, 1].grid(alpha=.2)

# C: phenotype is heart-rate driven
for ph, cc in [("A", "#1f6f4a"), ("B", "#c0392b")]:
    ax[1, 0].hist(ch[ch.phenotype == ph]["heart_rate_bpm"], bins=12, alpha=0.6,
                  color=cc, label=f"phenotype {ph}")
ax[1, 0].set_title("C. The phenotype is HEART-RATE driven,\nnot treatment (identical R-amplitude)",
                   fontweight="bold")
ax[1, 0].set_xlabel("heart rate (bpm)"); ax[1, 0].set_ylabel("animals")
ax[1, 0].legend(fontsize=9); ax[1, 0].grid(alpha=.2)

# D: the real signal is a continuous dose-response
dd = ch[ch.group != "Control"].copy()
dmap = {"Dox": 0, "Dox+Etn1.6": 1.6, "Dox+Etn16": 16, "Dox+Etn160": 160}
dd["ed"] = dd["group"].map(dmap)
jit = dd["ed"].rank(method="dense") + np.random.RandomState(0).uniform(-0.12, 0.12, len(dd))
ax[1, 1].scatter(jit, dd["r_amplitude_mv"], s=45, c="#8e44ad", edgecolor="k", linewidth=0.3)
mn = dd.groupby("ed")["r_amplitude_mv"].mean()
ax[1, 1].plot(range(1, len(mn) + 1), mn.values, "-o", color="#4a235a", lw=2, label="group mean")
rho, p = stats.spearmanr(dd["ed"], dd["r_amplitude_mv"])
ax[1, 1].set_xticks(range(1, 5)); ax[1, 1].set_xticklabels(["Dox\n(0)", "1.6", "16", "160"])
ax[1, 1].set_title(f"D. The real signal: continuous DOSE-RESPONSE\nR-amp declines with eth dose "
                   f"(Spearman {rho:.2f}, p={p:.3f})", fontweight="bold")
ax[1, 1].set_xlabel("ethanolamine dose (mg/kg)"); ax[1, 1].set_ylabel("R amplitude (mV)")
ax[1, 1].legend(fontsize=9); ax[1, 1].grid(alpha=.2)

fig.suptitle("Grouping: treatment groups don't cluster; the data groups by heart-rate phenotype; "
             "the treatment signal is a weak continuous dose-response",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
import figutil; figutil.explode(fig, "grouping_story")
print("saved outputs/figures/grouping_story.png")

## Normality / assumption-check figure

*Q–Q and distribution checks.*  
<sub>source: `normality_figure.py`</sub>

In [ ]:
"""Normality-check figures for the chronic clean set.
  normality_qq_grid.png  - Q-Q plot per parameter with Shapiro-Wilk p + verdict
  normality_ramp.png     - R-amplitude: histogram + normal curve + Q-Q plot
A Q-Q plot: points on the diagonal = normal; systematic curve away = non-normal.
"""
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

ALPHA = 0.05
m = pd.read_csv("../outputs/_metadata_merged.csv")
ch = m[(m.status == "OK") & (m.rr_cv <= 0.15) & (m.study == "chronic")].copy()
PARAMS = [("r_amplitude_mv", "R-amplitude (mV)"), ("t_wave_amplitude_mv", "T-amplitude (mV)"),
          ("p_wave_amplitude_mv", "P-amplitude (mV)"), ("qtc_ms", "QTc (ms)"),
          ("heart_rate_bpm", "Heart rate (bpm)"), ("qt_ms", "QT (ms)"),
          ("qrs_duration_ms", "QRS (ms)")]

# ---------- Q-Q grid for all parameters ----------
fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.ravel()
for ax, (col, name) in zip(axes, PARAMS):
    x = pd.to_numeric(ch[col], errors="coerce").dropna().values
    p = stats.shapiro(x).pvalue
    normal = p > ALPHA
    stats.probplot(x, dist="norm", plot=ax)
    ax.get_lines()[0].set(marker="o", markerfacecolor="#1f6f8f",
                          markeredgecolor="k", markersize=5, alpha=0.8)
    ax.get_lines()[1].set(color="#c0392b", lw=2)
    ax.set_title("%s\nShapiro p = %.3f  → %s" % (name, p, "NORMAL" if normal else "not normal"),
                 fontsize=10.5, fontweight="bold",
                 color=("#1a5e1a" if normal else "#a03030"))
    ax.set_xlabel("theoretical quantiles", fontsize=8)
    ax.set_ylabel("observed", fontsize=8); ax.grid(alpha=.2)
axes[-1].axis("off")
axes[-1].text(0.05, 0.7, "How to read a Q–Q plot:", fontweight="bold", fontsize=11, transform=axes[-1].transAxes)
axes[-1].text(0.05, 0.5, "• points ON the red line → normal\n"
                         "• points curving AWAY → not normal\n\n"
                         "Normal → parametric (ANOVA/Dunnett)\n"
                         "Not normal → non-parametric\n(Kruskal–Wallis/Dunn's)",
              fontsize=9.5, transform=axes[-1].transAxes, va="top")
fig.suptitle("Normality check — Q–Q plots per parameter (chronic clean set, n = %d)\n"
             "This is the visual basis for choosing parametric vs non-parametric tests" % len(ch),
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("../outputs/figures/normality_qq_grid.png", dpi=145, bbox_inches="tight")
plt.close(fig)
print("saved normality_qq_grid.png")

# ---------- R-amplitude focus: histogram + Q-Q ----------
x = pd.to_numeric(ch["r_amplitude_mv"], errors="coerce").dropna().values
p = stats.shapiro(x).pvalue
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.2))
a1.hist(x, bins=14, density=True, color="#aed6e6", edgecolor="k", alpha=0.85)
xs = np.linspace(x.min(), x.max(), 200)
a1.plot(xs, stats.norm.pdf(xs, x.mean(), x.std()), color="#c0392b", lw=2.5, label="normal curve")
a1.axvline(x.mean(), color="#2c3e50", ls="--", lw=1.5, label="mean")
a1.set_title("R-amplitude distribution vs a normal curve", fontweight="bold", fontsize=12)
a1.set_xlabel("R-wave amplitude (mV)"); a1.set_ylabel("density"); a1.legend(fontsize=9); a1.grid(alpha=.2)
stats.probplot(x, dist="norm", plot=a2)
a2.get_lines()[0].set(marker="o", markerfacecolor="#1f6f8f", markeredgecolor="k", markersize=6, alpha=0.8)
a2.get_lines()[1].set(color="#c0392b", lw=2)
a2.set_title("R-amplitude Q–Q plot", fontweight="bold", fontsize=12)
a2.set_xlabel("theoretical quantiles"); a2.set_ylabel("observed"); a2.grid(alpha=.2)
fig.suptitle("R-wave amplitude is approximately normal (Shapiro–Wilk p = %.2f > 0.05) — "
             "so parametric ANOVA / Dunnett are valid" % p, fontweight="bold", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig("../outputs/figures/normality_ramp.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved normality_ramp.png | R-amp Shapiro p = %.3f" % p)